# 📘 부록: RunPod GPU에서 실습하기

이 단원부터 실습 코드는 GPU에서 실행합니다. GPU 서버는 클라우드 서비스 **RunPod**에서 사용한 시간만큼 요금을 내고 빌려 쓰고, 수업에서 쓰던 **VS Code**로 원격 접속해 사용합니다.

**이 부록의 흐름**
1. 처음 한 번: SSH 키 쌍 생성, 공개키 등록, VS Code 확장 설치
2. 파드(GPU 서버) 생성
3. VS Code로 원격 접속
4. `git clone` 으로 실습 자료 복제, `uv sync` 로 파이썬 환경 구성, `uv run 업데이트.py` 로 한글 글꼴, 커널, VS Code 확장 준비
5. GPU 인식 확인과 교안 실행
6. 결과 내려받기, 파드 정지·종료

## 1. RunPod 기본

### 1-1. 용어

- **파드(Pod):** RunPod에서 빌려 쓰는 GPU 서버(컨테이너) 한 대입니다. 실행 중인 시간만큼 요금이 부과됩니다.
- **템플릿(Template):** 컨테이너 이미지와 기본 설정(노출 포트, 디스크 크기, 환경 변수)을 묶은 파드 구성입니다. 이 과정은 PyTorch·CUDA·git·uv가 설치된 공식 **PyTorch** 템플릿을 씁니다.

### 1-2. 저장 공간과 요금

| 저장 공간 | 정지(Stop) 시 | 종료(Terminate) 시 | 요금 |
|---|---|---|---|
| 컨테이너 디스크(기본 50GB) | 삭제 | 삭제 | 실행 중에만 |
| 네트워크 볼륨 | 유지 | 유지 | 삭제할 때까지 매월 |

- **이 단원은 컨테이너 디스크만 씁니다.** 정지하면 파일이 모두 삭제되므로 결과는 정지 전에 내려받습니다.
- **VS Code나 브라우저 창을 닫아도 파드는 계속 실행됩니다.** 실습이 끝나면 반드시 정지한 뒤 종료합니다.
- 파인튜닝 단원부터는 내려받은 모델을 보관하려고 네트워크 볼륨을 씁니다.

## 2. GPU 고르기

### 2-1. 딥러닝에 NVIDIA GPU를 쓰는 이유

딥러닝 계산은 대부분 **행렬 곱셈**이라, 많은 연산을 동시에 처리하는 GPU가 CPU보다 훨씬 빠릅니다.

- **병렬 연산:** 수천 개의 코어가 같은 계산을 동시에 처리합니다.
- **텐서 코어:** 행렬 곱셈 전용 연산 장치라 학습 속도가 더 빠릅니다.
- **HBM·NVLink:** 고대역폭 메모리에 큰 모델을 올리고, 여러 GPU를 고속으로 묶어 학습합니다.
- **CUDA 생태계:** PyTorch가 CUDA를 기본 지원해 `.to("cuda")` 한 줄로 GPU를 씁니다. NVIDIA가 사실상 표준이 된 가장 큰 이유입니다.

### 2-2. NVIDIA GPU 계열과 구입 가격

NVIDIA는 데이터센터 GPU의 공식 소비자가를 공개하지 않습니다. 아래 가격은 리셀러·OEM 견적과 시장 분석 자료를 모은 **1장 기준 대략값** 이며, 1달러 약 1,330원으로 환산했습니다. 공급 상황과 환율에 따라 크게 바뀝니다. 표는 구입 가격이 낮은 순서입니다.

| 구분 | 아키텍처(출시) | 모델 | GPU 메모리 | 구입 가격(대략) |
|---|---|---|---|---:|
| 소비자(게이밍) | Blackwell(2025) | RTX 5090 | 32GB GDDR7 | 530만~670만 원(출시가 약 270만 원) |
| 데이터센터 | Ampere(2020) | A100 | 40·80GB HBM2e | 중고 530만~1,200만 원, 신품 1,330만~2,000만 원 |
| 워크스테이션 | Blackwell(2025) | RTX PRO 6000 | 96GB GDDR7 | 1,600만~2,100만 원 |
| 데이터센터 | Hopper(2022) | H100 | 80GB HBM3 | 3,300만~5,300만 원 |
| 데이터센터 | Hopper(2024) | H200 | 141GB HBM3e | 4,000만~5,300만 원(견적에 따라 6,700만 원 이상) |
| 데이터센터 | Blackwell(2025) | B200 | 180~192GB HBM3e | 4,000만~6,700만 원 |
| 데이터센터 | Blackwell Ultra(2025) | B300 | 288GB HBM3e | 6,700만~8,000만 원 |
| 데이터센터 | Rubin(2026 출하 시작) | Rubin GPU | 288GB HBM4 | 약 7,300만 원(대량 구매, 증권사 추정) |
| 데이터센터 | Blackwell(2024) | B100 | 192GB HBM3e | 가격 없음(초기 발표 모델, 패키징 문제로 사실상 B200에 흡수) |

- **데이터센터 GPU는 보통 8장 서버 단위로 판매됩니다.** H100 8장 서버는 약 3.3억~4.7억 원, DGX B300(8장)은 약 4억~4.7억 원, Rubin 72장 랙(NVL72)은 약 66억~117억 원으로 추정됩니다.
- **데이터센터용과 게이밍용의 차이:** 데이터센터용은 메모리가 크고(HBM), NVLink로 여러 장을 묶어 학습할 수 있습니다. 게이밍용은 값이 싸지만 메모리가 작아 큰 모델 학습에는 한계가 있습니다.
- **그래서 이 과정은 GPU를 사지 않고 빌려 씁니다.** 16GB GPU는 시간당 약 330원, H100도 시간당 수천 원 수준이라 실습한 시간만큼만 비용이 듭니다.
- 가격 출처: [IntuitionLabs 데이터센터 GPU 가격](https://intuitionlabs.ai/articles/nvidia-ai-gpu-pricing-guide), [Thunder Compute B200](https://www.thundercompute.com/blog/nvidia-b200-pricing)·[B300](https://www.thundercompute.com/blog/nvidia-b300-pricing), [Tom's Hardware Vera Rubin NVL72](https://www.tomshardware.com/tech-industry/artificial-intelligence/price-of-nvidias-vera-rubin-nvl72-racks-skyrockets-to-as-much-as-usd8-8-million-apiece-but-server-makers-margins-will-be-tight-nvidia-is-moving-closer-to-shipping-entire-full-scale-systems), [TechSpot RTX PRO 6000](https://www.techspot.com/news/113460-nvidia-raises-rtx-pro-6000-blackwell-price-staggering.html), [JarvisLabs A100](https://jarvislabs.ai/blog/a100-price)

### 2-3. 이 과정의 권장 GPU

가장 먼저 **GPU 메모리(VRAM)** 를 확인합니다. 모델 가중치와 학습 중 계산 값(배치 데이터, 기울기, 옵티마이저 상태)이 GPU 메모리에 모두 올라가야 학습이 진행되고, 부족하면 `CUDA out of memory` 오류로 멈춥니다.

| 단계 | 권장 메모리 | RunPod GPU 예(시간당 대략) |
|---|---|---:|
| 딥러닝 학습 원리·PyTorch·트랜스포머 | 16GB | RTX 2000 Ada·RTX A4000(0.25달러, 약 330원) |
| 파인튜닝(SFT·LoRA·QLoRA·DPO) | 24GB | RTX A5000(0.27달러, 약 360원)·RTX 4090(0.9달러, 약 1,200원) |
| 서빙(vLLM)·벤치마크 | 48GB | A40·RTX A6000(0.6달러, 약 800원) |

- **이 단원은 16GB면 충분합니다.**
- 저렴한 GPU는 재고가 없을 때가 많습니다(**Out of capacity**). **Available** 탭에는 지금 빌릴 수 있는 GPU만 표시됩니다.
- NVIDIA GPU를 고릅니다. AMD GPU(MI300X 등)는 CUDA 대신 ROCm을 써서 이 과정이 설치하는 PyTorch(CUDA 12.8 빌드)가 동작하지 않습니다.

## 3. 처음 한 번만: SSH 키와 VS Code 준비

**SSH(Secure Shell)** 는 네트워크로 원격 서버에 암호화된 연결을 맺고 명령을 실행하는 프로토콜입니다. VS Code의 **Remote - SSH** 확장은 이 연결을 통해 서버의 파일을 편집하고 터미널과 노트북을 서버에서 실행합니다.

접속할 때는 비밀번호 대신 **SSH 키 쌍**으로 본인을 인증합니다.

- **개인키**(`~/.ssh/id_ed25519`): 본인임을 증명하는 키입니다. 내 PC에만 보관하고 누구와도 공유하지 않습니다.
- **공개키**(`~/.ssh/id_ed25519.pub`): 서버에 등록하는 키입니다. RunPod는 등록된 공개키를 파드에 넣어 두고, 짝이 되는 개인키로 인증한 연결만 허용합니다.

이 절은 처음 한 번만 합니다.

### 3-1. SSH 키 쌍 만들기

VS Code에서 **Terminal → New Terminal** 로 내 PC의 터미널을 엽니다. macOS(zsh)와 Windows(PowerShell)에서 같은 명령을 씁니다.

**OpenSSH 명령(내 PC 터미널)**

```bash
# SSH 키 쌍 생성
ssh-keygen -t ed25519 -C runpod
```

- `ssh-keygen`: SSH 키 쌍을 만드는 OpenSSH 프로그램
- `-t ed25519`: 키 알고리즘을 Ed25519로 지정합니다(짧고 안전한 최신 방식).
- `-C runpod`: 키에 붙는 설명(comment)입니다. 여러 키를 구분하는 이름표 역할만 합니다.

저장 위치, 암호(passphrase), 암호 확인을 차례로 묻습니다. 세 번 모두 Enter를 눌러 기본값(저장 위치 `~/.ssh/`, 암호 없음)으로 만듭니다. `Overwrite (y/n)?` 가 나오면 키가 이미 있다는 뜻이니 `n` 을 누르고 기존 키를 씁니다.

**셸 명령(내 PC 터미널)**

```bash
# 공개키 파일 내용 출력
cat ~/.ssh/id_ed25519.pub
```

- `cat`: 파일 내용을 터미널에 출력합니다.
- `~/.ssh/id_ed25519.pub`: 방금 만든 공개키 파일입니다(`~` 는 내 사용자 폴더).

출력된 한 줄(`ssh-ed25519` 로 시작) 전체를 복사합니다.

<img src="images/vscode_01_keygen.png" width="1050" alt="VS Code 터미널에서 ssh-keygen 을 실행한 화면. Enter로 넘긴 질문 세 줄과 cat 으로 출력한 공개키 한 줄에 빨간 사각형이 그려져 있다.">

### 3-2. RunPod에 공개키 등록

공개키는 **파드를 만들기 전에** 등록합니다. 파드를 만들 때 등록된 공개키가 파드에 들어가므로, 이미 만든 파드에는 나중에 등록한 키가 적용되지 않습니다.

1. 왼쪽 메뉴 **Credentials** → **SSH Public Keys** 탭 → **Add SSH key**
2. **Title** 에 키 이름(예: `my-laptop`), **Key** 에 복사한 공개키를 붙여 넣고 **Add SSH key**

<img src="images/runpod_10_ssh_keys.png" width="960" alt="RunPod 콘솔 Credentials 화면. 왼쪽 메뉴 Credentials, SSH Public Keys 탭, Add SSH key 버튼에 빨간 사각형이 그려져 있다.">

<img src="images/runpod_11_add_key.png" width="560" alt="Add SSH key 창. Key 칸에 붙여 넣은 공개키와 Add SSH key 버튼에 빨간 사각형이 그려져 있다.">

### 3-3. Remote - SSH 확장 설치

VS Code 왼쪽 **Extensions** 에서 `Remote - SSH` 를 검색하고, 게시자가 Microsoft인 확장을 **Install** 합니다.

<img src="images/vscode_02_remote_ssh.png" width="720" alt="VS Code 확장 검색 화면. 검색 칸의 Remote - SSH 와 Microsoft Remote - SSH 확장의 Install 버튼에 빨간 사각형이 그려져 있다.">

## 4. 파드 만들기

1. 왼쪽 메뉴 **Billing** 에서 크레딧을 충전합니다. 크레딧이 있어야 파드를 만들 수 있습니다.
2. **Pods** 를 누르면 **Deploy a Pod** 화면이 열립니다. **Workload** 에 **PyTorch** 템플릿이 선택되어 있는지 확인하고, **Pod name** 을 입력합니다(예: `day65-practice`).

<img src="images/runpod_01_pods.png" width="1050" alt="RunPod 콘솔의 Deploy a Pod 화면. 왼쪽 메뉴의 Pods와 화면 제목 Deploy a Pod에 빨간 사각형이 그려져 있고, 아래에 PyTorch 템플릿과 Pod name 칸이 보인다.">

3. **Set overrides** 를 눌러 아래 설정이 켜져 있는지 확인하고 창을 닫습니다.
   - **SSH terminal access** 체크와 **Exposed TCP ports** 의 `22`(SSH 포트): VS Code SSH 접속에 필요합니다.
   - **Start Jupyter notebook** 체크: 9절 Jupyter Lab 접속에 필요합니다.

<img src="images/runpod_02_template.png" width="560" alt="Edit template overrides 창. Start Jupyter notebook 과 SSH terminal access 체크 상자, Exposed TCP ports 의 22 에 빨간 사각형이 그려져 있다.">

4. **Search GPUs** 에 `RTX 2000 Ada` 를 입력하고 **Available** 탭에서 GPU를 선택합니다.

<img src="images/runpod_03_gpu.png" width="875" alt="Compute 칸의 GPU 목록. 검색 칸의 RTX 2000 Ada, Available 탭, 고른 GPU 카드(RTX 2000 Ada, 16GB, 시간당 0.24달러)에 빨간 사각형이 그려져 있다.">

5. 저장 공간은 기본값(Container disk 50GB, Persistent storage 없음)으로 두고, **Total / hour**(시간당 총 요금)를 확인한 뒤 **Deploy Pod** 를 누릅니다.
   - 버튼이 **Deploy when available** 로 바뀌어 있으면 고른 GPU의 재고가 없다는 뜻입니다. 다른 GPU를 고릅니다.

<img src="images/runpod_04_summary.png" width="320" alt="배포 직전 오른쪽 요약 칸. Total / hour 0.25달러와 Deploy Pod 버튼에 빨간 사각형이 그려져 있다.">

## 5. VS Code로 파드에 접속하기

### 5-1. SSH 접속 명령 복사

파드 상태가 실행 중(Running)이 되면(1~2분) 상세 칸의 **Connect** 탭에서 **SSH over exposed TCP** 아래 명령을 복사합니다.

**SSH 접속 명령 예(직접 실행하지 않고 5-2에서 VS Code에 붙여 넣습니다)**

```bash
ssh root@38.147.83.27 -p 23887 -i ~/.ssh/id_ed25519
```

- `ssh`: SSH로 원격 서버에 접속하는 프로그램
- `root@38.147.83.27`: 접속할 사용자(`root`)와 파드의 IP 주소
- `-p 23887`: 접속 포트 번호입니다. IP와 포트는 파드마다 다릅니다.
- `-i ~/.ssh/id_ed25519`: 인증에 쓸 개인키 파일

위쪽 **SSH** 칸의 명령(`...@ssh.runpod.io`)은 RunPod 프록시를 거치는 연결입니다. 화면에 적힌 대로 파일 전송(SCP·SFTP)을 지원하지 않는 등 기능이 제한되어 VS Code 접속에는 쓸 수 없습니다.

<img src="images/runpod_05_connect.png" width="598" alt="실행 중인 파드의 상세 칸. Connect 탭과 SSH over exposed TCP 제목부터 접속 명령까지에 빨간 사각형이 그려져 있다.">

### 5-2. VS Code에 SSH 호스트 등록

1. 명령 팔레트(macOS `Cmd+Shift+P`, Windows `Ctrl+Shift+P`)에서 **Remote-SSH: Connect to Host...** 를 고릅니다.
2. **+ Add New SSH Host...** 를 고릅니다.
3. 복사한 접속 명령을 붙여 넣고 Enter를 누릅니다.
4. 저장할 SSH 설정 파일로 첫 번째(`~/.ssh/config`)를 고릅니다. 접속 정보가 이 파일에 저장됩니다.

<img src="images/vscode_03_connect_host.png" width="640" alt="명령 팔레트에 Remote-SSH: Connect to Host 를 입력한 화면. 입력 칸과 Remote-SSH: Connect to Host... 항목에 빨간 사각형이 그려져 있다.">

<img src="images/vscode_04_add_host.png" width="640" alt="Add New SSH Host... 항목에 빨간 사각형이 그려진 화면.">

<img src="images/vscode_05_ssh_command.png" width="640" alt="Enter SSH Connection Command 입력 칸에 ssh root@주소 -p 포트 -i ~/.ssh/id_ed25519 명령을 붙여 넣은 화면.">

<img src="images/vscode_06_config_file.png" width="640" alt="Select SSH configuration file to update 목록. 첫 번째 사용자 폴더의 .ssh/config 에 빨간 사각형이 그려져 있다.">

### 5-3. 접속하고 작업 폴더 열기

1. 다시 **Remote-SSH: Connect to Host...** 에서 등록한 파드 IP를 고릅니다. 원격 창이 새로 열립니다.
2. 처음 접속하면 `Are you sure you want to continue connecting (yes/no/[fingerprint])?` 가 나옵니다. 처음 접속하는 서버의 신원(호스트 키)을 확인하는 질문입니다. `yes` 를 입력하고 Enter를 누릅니다(입력 내용은 가려져 표시됩니다).
3. 왼쪽 아래 상태 표시줄에 `SSH: IP` 가 보이면 접속된 것입니다. 첫 접속은 파드에 VS Code 서버를 설치하므로 1분 정도 걸립니다.
4. **File → Open Folder...** 에 `/workspace/` 를 입력하고 **OK** 를 누릅니다. 작성자를 신뢰하는지 물으면 **Yes, I trust the authors** 를 누릅니다.

<img src="images/vscode_07_pick_host.png" width="640" alt="Connect to Host 목록. 새로 추가된 파드 주소에 빨간 사각형이 그려져 있다.">

<img src="images/vscode_08_open_folder.png" width="640" alt="원격 창의 Open Folder 입력 칸. /workspace/ 와 OK 버튼에 빨간 사각형이 그려져 있다.">

### 5-4. 파드의 Python·Jupyter 확장

Python·Jupyter 확장은 코드를 실행하는 쪽(파드)에서 동작하므로 파드에도 설치되어 있어야 합니다. 6-3의 `uv run 업데이트.py` 를 VS Code 터미널에서 실행하면 두 확장을 파드에 자동으로 설치하므로 지금은 넘어가도 됩니다.

직접 설치하려면 원격 창의 Extensions 에서 `Python` 을 검색해 **Install in SSH: IP** 를 누르고, `Jupyter` 도 같은 방법으로 설치합니다.

<img src="images/vscode_09_remote_ext.png" width="560" alt="접속한 창의 확장 검색 화면. 검색 칸의 Python 과 Python 확장의 Install in SSH 버튼에 빨간 사각형이 그려져 있다.">

## 6. 실습 자료 받기와 파이썬 환경 만들기

원격 창에서 **Terminal → New Terminal** 을 열면 **파드의 리눅스 터미널(bash)** 이 열립니다. 이 절의 명령은 모두 이 터미널에서 실행합니다.

| 명령 종류 | 하는 일 | 이 절의 명령 |
|---|---|---|
| 리눅스 셸 명령 | 폴더 이동, 파일 목록 확인 | `cd`, `ls` |
| git 명령 | 원격 저장소 복제 | `git clone` |
| uv 명령 | 파이썬 가상 환경 구성, 그 환경에서 실행 | `uv sync`, `uv run` |

### 6-1. 실습 자료 받기

**리눅스 셸 명령(파드 터미널)**

```bash
# 작업 폴더로 이동
cd /workspace
```

- `cd`: 현재 작업 폴더를 바꿉니다.
- `/workspace`: 파드의 작업용 폴더

**git 명령(파드 터미널)**

```bash
# 실습 저장소 복제
git clone https://github.com/paircodingofficial-cloud/enkoa-practice-deep-learning.git
```

- `git clone`: 원격 저장소의 파일과 이력을 현재 폴더 안의 새 폴더로 복제합니다.
- 주소: 과정 실습 저장소입니다. 같은 이름의 폴더 `enkoa-practice-deep-learning` 이 생깁니다.

**리눅스 셸 명령(파드 터미널)**

```bash
# 복제한 폴더의 파일 목록 확인
ls enkoa-practice-deep-learning
```

- `ls`: 폴더 안의 파일 목록을 출력합니다. `pyproject.toml`, `uv.lock`, 단원 폴더가 보이면 정상입니다.

<img src="images/vscode_10_clone.png" width="1050" alt="파드 터미널에서 git clone 을 실행한 화면. git clone 명령과 ls 로 본 저장소 파일 목록, 왼쪽 탐색기의 enkoa-practice-deep-learning 폴더에 빨간 사각형이 그려져 있다.">

### 6-2. 파이썬 환경 만들기

**리눅스 셸 명령(파드 터미널)**

```bash
# 저장소 폴더로 이동
cd enkoa-practice-deep-learning
```

- `cd enkoa-practice-deep-learning`: 복제한 저장소 폴더로 이동합니다. 이후 명령은 이 폴더 기준으로 실행됩니다.

**uv 명령(파드 터미널)**

```bash
# 가상 환경 생성과 패키지 설치
uv sync
```

- `uv sync`: `pyproject.toml`(필요한 패키지)과 `uv.lock`(정확한 버전)에 적힌 대로 가상 환경 `.venv` 를 만들고 패키지를 설치합니다. PyTorch는 CUDA 12.8용 빌드로 설치되며 1분쯤 걸립니다.

**uv 명령(파드 터미널)**

```bash
# PyTorch 버전과 GPU 사용 가능 여부 확인
uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
```

- `uv run`: 뒤의 명령을 이 프로젝트의 가상 환경(`.venv`)에서 실행합니다.
- `python -c "..."`: 따옴표 안의 파이썬 코드를 파일 없이 바로 실행합니다.
- 결과 `2.11.0+cu128 True`: PyTorch 버전(CUDA 12.8 빌드)과 GPU 사용 가능 여부입니다. `True` 면 준비가 끝났습니다.

<img src="images/vscode_11_uv_sync.png" width="1050" alt="uv sync 가 끝난 터미널. 설치된 torch==2.11.0+cu128 줄, uv run 으로 출력한 2.11.0+cu128 True, 탐색기의 .venv 폴더에 빨간 사각형이 그려져 있다.">

### 6-3. 한글 글꼴, Jupyter 커널, VS Code 확장 준비

**uv 명령(파드 터미널, 저장소 폴더)**

```bash
# 최신 자료 받기, 패키지 맞추기, 한글 글꼴 복사, Jupyter 커널 등록, VS Code 확장 설치
uv run 업데이트.py
```

- `uv run 업데이트.py`: 저장소에 들어 있는 도우미 스크립트를 `.venv` 에서 실행합니다. 차례로 아래 일을 합니다.
  1. 서버의 최신 자료로 맞춥니다. 원본을 고쳐 둔 파일은 먼저 `.백업/` 에 보관합니다.
  2. `uv sync` 로 패키지를 맞춥니다. 6-2에서 이미 설치했으므로 바로 넘어갑니다.
  3. 저장소의 나눔고딕 글꼴(`fonts/`)을 파드의 글꼴 폴더(`~/.fonts`)에 복사합니다. 그래프에 한글이 표시됩니다.
  4. `.venv` 를 Jupyter Lab 커널 **venv** 로 등록합니다(9절에서 씁니다).
  5. VS Code 터미널에서 실행했다면 Python·Jupyter 확장을 파드에 설치합니다(5-4).
- 이 도우미는 RunPod(리눅스) 전용입니다. 내 PC에서 실행하면 아무것도 하지 않고 끝납니다.

### 6-4. 저장소 폴더 열기

**File → Open Folder...** 에 `/workspace/enkoa-practice-deep-learning/` 을 입력해 엽니다. VS Code는 연 폴더 안의 `.venv` 를 찾아 커널 목록에 보여 줍니다. `/workspace` 를 연 상태에서는 저장소 안의 `.venv` 가 목록에 나오지 않습니다.

<img src="images/vscode_12_open_repo.png" width="640" alt="Open Folder 입력 칸에 /workspace/enkoa-practice-deep-learning/ 을 넣은 화면. 입력 칸과 OK 버튼에 빨간 사각형이 그려져 있다.">

### 6-5. 다음 파드부터: 한 줄로 준비하기

두 번째 파드부터는 6-1~6-3을 아래 한 줄로 대신합니다. 이미 저장소가 있으면 복제는 건너뛰고 최신 자료로 맞춥니다.

**리눅스 셸 명령(파드 터미널)**

```bash
# 저장소가 없으면 복제하고, 저장소 폴더에서 업데이트 도우미 실행
cd /workspace && { [ -d enkoa-practice-deep-learning ] || git clone https://github.com/paircodingofficial-cloud/enkoa-practice-deep-learning.git; } && cd enkoa-practice-deep-learning && uv run 업데이트.py
```

- `&&`: 앞 명령이 성공하면 뒤 명령을 이어서 실행합니다.
- `[ -d 폴더 ]`: 폴더가 있으면 참입니다. `||` 는 앞이 거짓일 때만 뒤를 실행하므로, 저장소 폴더가 없을 때만 `git clone` 합니다.
- `{ ...; }`: 안의 명령을 한 덩어리로 묶습니다.

명령이 끝나면 6-4대로 저장소 폴더를 엽니다.

## 7. GPU 확인하고 교안 실행하기

### 7-1. 노트북 만들고 커널 고르기

**커널(kernel)** 은 노트북 셀의 코드를 받아 실행하는 프로그램입니다. 어느 파이썬 환경의 커널을 고르느냐에 따라 쓸 수 있는 패키지가 달라집니다.

1. 명령 팔레트에서 **Create: New Jupyter Notebook** 으로 새 노트북을 만들고 `내작업/GPU_확인.ipynb` 로 저장합니다.
2. 오른쪽 위 **Select Kernel** → **Python Environments...** 를 고릅니다.
3. **.venv (Python 3.12.3)** 를 고릅니다(`Recommended` 표시).

목록의 다른 파이썬(3.10, `/usr/bin/python3` 등)은 템플릿에 원래 설치된 파이썬이라 이 저장소가 쓰는 패키지(pandas 등)가 없습니다. 반드시 `.venv` 를 고릅니다.

<img src="images/vscode_13_select_kernel.png" width="1050" alt="노트북 오른쪽 위 Select Kernel 버튼과 커널 종류 목록의 Python Environments... 에 빨간 사각형이 그려져 있다.">

<img src="images/vscode_14_venv_kernel.png" width="640" alt="Select a Python Environment 목록. Recommended 표시가 붙은 .venv (Python 3.12.3) 에 빨간 사각형이 그려져 있다.">

### 7-2. GPU 인식 확인

아래 두 코드를 각각 셀에 넣어 실행합니다.

**노트북 셀(`.venv` 커널, 파드에서 실행)**

```python
# GPU 이름과 메모리, 드라이버가 지원하는 CUDA 버전을 확인합니다
!nvidia-smi
```

- `!`: 노트북 셀에서 셸 명령을 실행하라는 표시입니다.
- `nvidia-smi`: NVIDIA 드라이버에 들어 있는 GPU 상태 조회 도구입니다. GPU 이름, 메모리, 사용률을 보여 줍니다.

**노트북 셀(`.venv` 커널, 파드에서 실행)**

```python
import torch

# PyTorch가 GPU를 쓸 수 있는지, 어떤 GPU인지 확인합니다
print("PyTorch 버전:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print("GPU 이름:", torch.cuda.get_device_name(0), "| GPU 메모리(GB):", round(total_gb, 1))
```

- `torch.cuda.is_available()`: PyTorch가 GPU(CUDA)를 쓸 수 있으면 `True`
- `torch.cuda.get_device_properties(0).total_memory`: 0번 GPU의 전체 메모리(바이트). `/ 1024 ** 3` 으로 GB 단위로 바꿉니다.
- `torch.cuda.get_device_name(0)`: 0번 GPU의 이름

`GPU 사용 가능: True` 이고 버전이 `2.11.0+cu128` 이면 정상입니다.
- 버전이 `2.8` 로 시작하면 템플릿 파이썬 커널을 고른 것입니다. 7-1대로 `.venv` 로 바꿉니다.
- 7절과 8절은 RTX A6000 파드에서 캡처한 화면이라 GPU 이름·메모리·요금이 다르게 보일 수 있습니다.

<img src="images/vscode_15_gpu_check.png" width="1050" alt="VS Code에서 nvidia-smi 와 torch.cuda 확인 셀을 실행한 화면. 커널 .venv (Python 3.12.3), GPU 이름 NVIDIA RTX A6000, 메모리 49140MiB, 출력 GPU 사용 가능: True 에 빨간 사각형이 그려져 있다.">

### 7-3. 교안 실행

`day65_딥러닝_학습원리/교안_01_손실함수와_경사하강.ipynb` 를 열고 `.venv` 커널로 위에서부터 실행합니다. 준비 셀이 `계산 장치: cuda | (GPU 이름)` 을 출력하면 GPU로 계산하고 있는 것입니다.

<img src="images/vscode_16_lesson_cuda.png" width="1050" alt="VS Code에서 교안 01 준비 셀과 데이터 읽기 셀을 실행한 화면. 커널 .venv (Python 3.12.3), 출력 계산 장치: cuda | NVIDIA RTX A6000, 행 수: 100 에 빨간 사각형이 그려져 있다.">

## 8. 끝내기: 결과 내려받기와 정지·종료

1. **내려받기:** 탐색기에서 받을 폴더(단원 폴더, `내작업`)를 오른쪽 클릭 → **Download...** → 내 PC의 저장 위치를 고릅니다. 폴더 전체를 내려받습니다.

<img src="images/vscode_17_download.png" width="740" alt="VS Code 탐색기에서 단원 폴더를 오른쪽 클릭한 메뉴. 폴더와 Download... 항목에 빨간 사각형이 그려져 있다.">

2. **정지:** RunPod **Pods** 화면에서 파드를 눌러 상세 칸을 열고 **Stop** → **Stop Pod** 를 누릅니다. 볼륨이 없으면 `ALL DATA will be lost!`(모든 데이터 삭제) 경고가 표시됩니다.

<img src="images/runpod_08_stop.png" width="560" alt="정지 확인 창. 볼륨이 없어 모든 데이터가 지워진다는 경고와 Stop Pod 버튼에 빨간 사각형이 그려져 있다.">

3. **종료:** 정지되면 나타나는 **Terminate** → **Terminate Pod** 를 누릅니다. 실행 중인 파드에는 Terminate 버튼이 없습니다. 마지막으로 Pods 목록이 비었는지 확인합니다.

<img src="images/runpod_09_terminate.png" width="600" alt="정지된 파드의 상세 칸. 정지한 뒤 나타난 Terminate 버튼에 빨간 사각형이 그려져 있다.">

**다음에 새 파드로 실습할 때:** 4절로 파드를 만든 뒤 5절부터 다시 합니다. 파드마다 IP와 포트가 바뀌기 때문입니다. 6절은 6-5의 한 줄 명령으로 대신합니다. 3절(SSH 키, 공개키 등록, Remote - SSH 확장)은 다시 하지 않습니다.

## 9. VS Code 접속이 안 될 때: Jupyter Lab

브라우저의 Jupyter Lab에서도 같은 `.venv` 환경으로 실습할 수 있습니다. VS Code와 달리 Jupyter Lab은 `.venv` 를 스스로 찾지 못하므로, 커널로 한 번 등록합니다.

1. 파드 상세 칸 **Connect** 탭에서 **Jupyter Lab** 이 **Ready** 가 되면 링크를 엽니다.
2. **File → New → Terminal** 에서 6-1, 6-2, 6-3을 차례로 실행합니다(두 번째 파드부터는 6-5의 한 줄 명령). 6-3의 `uv run 업데이트.py` 가 `.venv` 를 커널 **venv** 로 등록합니다.
3. 브라우저를 새로 고침하고 교안을 연 뒤, 커널로 **venv** 를 고릅니다.

<img src="images/runpod_05b_jupyter.png" width="598" alt="파드 상세 칸의 Connect 탭. HTTP services 의 Jupyter Lab 링크와 Ready 표시에 빨간 사각형이 그려져 있다.">

## 10. 자주 막히는 곳

| 증상 | 원인과 해결 |
|---|---|
| 접속할 때 비밀번호를 묻는다 | 공개키가 파드에 없습니다. 3-2로 공개키를 등록한 뒤 파드를 새로 만듭니다 |
| `Connection timed out` | 파드가 아직 시작 중입니다. 1분 뒤 다시 접속합니다. Connect 탭에 **SSH over exposed TCP** 가 없으면 4절 3번 설정을 켜고 파드를 새로 만듭니다 |
| `REMOTE HOST IDENTIFICATION HAS CHANGED` | 예전에 쓴 IP·포트가 새 파드에 다시 배정됐습니다. 아래 OpenSSH 명령으로 옛 호스트 키를 지우고 다시 접속합니다 |
| 커널 목록에 `.venv` 가 없다 | `/workspace` 를 연 상태입니다. 6-4대로 저장소 폴더를 엽니다 |
| `No module named 'pandas'` | 커널이 `.venv` 가 아닙니다. 7-1대로 바꿉니다(Jupyter Lab은 9절의 **venv**) |
| 준비 셀이 `cpu` 를 출력한다 | GPU 없이 만든 파드이거나 드라이버가 낮습니다. `nvidia-smi` 의 CUDA Version이 12.8 이상인지 보고, 아니면 다른 GPU로 새로 만듭니다 |
| `CUDA out of memory` | GPU 메모리가 부족합니다. 배치 크기를 줄이거나 메모리가 큰 GPU로 새로 만듭니다 |
| 그래프 한글이 네모로 나온다 | 6-3의 `uv run 업데이트.py` 를 실행한 뒤 노트북 **Restart** 로 커널을 다시 시작합니다 |
| `git` 또는 `uv` 가 없다는 오류 | 아래 명령으로 설치합니다(PyTorch 템플릿에는 이미 설치되어 있습니다) |

**OpenSSH 명령(내 PC 터미널)**

```bash
# 저장된 옛 호스트 키 삭제(IP와 포트는 경고에 나온 값으로 바꾸고, 대괄호와 따옴표는 그대로 둡니다)
ssh-keygen -R "[38.147.83.27]:23887"
```

- `-R`: `~/.ssh/known_hosts`(접속했던 서버의 호스트 키 목록)에서 그 서버의 기록을 지웁니다.
- `"[IP]:포트"`: 22번이 아닌 포트로 접속한 서버는 이 형식으로 저장됩니다.

**apt 명령(파드 터미널)**

```bash
# git 과 curl 설치
apt-get update && apt-get install -y git curl
```

- `git`: 저장소 복제 도구, `curl`: 인터넷 주소의 파일을 내려받는 도구
- `apt-get update`: 설치할 수 있는 패키지 목록을 최신으로 갱신합니다.
- `apt-get install -y`: 패키지를 설치합니다. `-y` 는 설치 확인 질문에 자동으로 yes를 답합니다.
- `&&`: 앞 명령이 성공하면 뒤 명령을 이어서 실행합니다.

**리눅스 셸 명령(파드 터미널)**

```bash
# uv 설치 스크립트 실행
curl -LsSf https://astral.sh/uv/install.sh | sh
```

- `curl -LsSf 주소`: 주소의 설치 스크립트를 내려받습니다. `-L` 주소가 옮겨졌으면 따라감, `-s` 진행 표시와 메시지 숨김, `-S` 그래도 오류 메시지는 표시, `-f` HTTP 오류(404 등)면 오류 페이지를 넘기지 않고 실패로 끝냄(오류 페이지가 `sh` 로 실행되는 것을 막음).
- `| sh`: 내려받은 스크립트를 셸에 넘겨 바로 실행합니다.

**리눅스 셸 명령(파드 터미널)**

```bash
# 방금 설치한 uv 경로를 현재 터미널에 적용
source $HOME/.local/bin/env
```

- `source 파일`: 파일에 적힌 셸 명령을 현재 터미널에서 실행합니다. 이 파일은 uv가 있는 `~/.local/bin` 을 PATH(명령을 찾는 폴더 목록)에 추가합니다. 새로 연 터미널에는 자동으로 적용됩니다.

## 11. 핵심 정리

- **처음 한 번:** SSH 키 쌍 생성 → 공개키 등록 → Remote - SSH 확장 설치
- **첫 파드:** 파드 생성 → 접속 → `git clone` → `uv sync` → `uv run 업데이트.py` → 저장소 폴더 열기 → `.venv` 커널
- **다음 파드부터:** 파드 생성 → 접속 → 6-5의 한 줄 명령 → 저장소 폴더 열기 → `.venv` 커널
- **끝나면:** 내려받기 → 정지 → 종료